<a href="https://colab.research.google.com/github/vishmitha99/EEX5362-Performance-Modelling-Case-Study/blob/main/Nexora_Lab01_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from google.colab import files
import zipfile
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
spark = (
    SparkSession.builder
    .appName("Nexora COVID-19 Analysis")
    .getOrCreate()
)

print("Spark session created successfully!")
spark

Spark session created successfully!


In [4]:
uploaded = files.upload()

Saving archive.zip to archive.zip


In [6]:
#Extract the dataset
zip_file = "archive.zip"

with zipfile.ZipFile(zip_file, "r") as z:
    z.extractall("/content/covid_data")

print("Dataset extracted successfully!")

os.listdir("/content/covid_data")

Dataset extracted successfully!


['worldometer_coronavirus_summary_data.csv',
 'worldometer_coronavirus_daily_data.csv']

In [8]:
#Load the daily dataset into PySpark
file_path = (
    "/content/covid_data/"
    "worldometer_coronavirus_daily_data.csv"
)

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [9]:
#Display the column names and data types
df.printSchema()

root
 |-- date: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- cumulative_total_cases: double (nullable = true)
 |-- daily_new_cases: double (nullable = true)
 |-- active_cases: double (nullable = true)
 |-- cumulative_total_deaths: double (nullable = true)
 |-- daily_new_deaths: double (nullable = true)



In [10]:
#Display the first five records
df.show(5, truncate=False)

+-------------------+-----------+----------------------+---------------+------------+-----------------------+----------------+
|date               |country    |cumulative_total_cases|daily_new_cases|active_cases|cumulative_total_deaths|daily_new_deaths|
+-------------------+-----------+----------------------+---------------+------------+-----------------------+----------------+
|2020-02-15 00:00:00|Afghanistan|0.0                   |NULL           |0.0         |0.0                    |NULL            |
|2020-02-16 00:00:00|Afghanistan|0.0                   |NULL           |0.0         |0.0                    |NULL            |
|2020-02-17 00:00:00|Afghanistan|0.0                   |NULL           |0.0         |0.0                    |NULL            |
|2020-02-18 00:00:00|Afghanistan|0.0                   |NULL           |0.0         |0.0                    |NULL            |
|2020-02-19 00:00:00|Afghanistan|0.0                   |NULL           |0.0         |0.0                    |NU

In [11]:
#Count the rows
print("Total rows:", df.count())

print("Total columns:", len(df.columns))

print("Number of countries/locations:", df.select("country").distinct().count())

Total rows: 184787
Total columns: 7
Number of countries/locations: 226


In [15]:
#Operation 1 — Display records using show()
df.show(10)

+-------------------+-----------+----------------------+---------------+------------+-----------------------+----------------+
|               date|    country|cumulative_total_cases|daily_new_cases|active_cases|cumulative_total_deaths|daily_new_deaths|
+-------------------+-----------+----------------------+---------------+------------+-----------------------+----------------+
|2020-02-15 00:00:00|Afghanistan|                   0.0|           NULL|         0.0|                    0.0|            NULL|
|2020-02-16 00:00:00|Afghanistan|                   0.0|           NULL|         0.0|                    0.0|            NULL|
|2020-02-17 00:00:00|Afghanistan|                   0.0|           NULL|         0.0|                    0.0|            NULL|
|2020-02-18 00:00:00|Afghanistan|                   0.0|           NULL|         0.0|                    0.0|            NULL|
|2020-02-19 00:00:00|Afghanistan|                   0.0|           NULL|         0.0|                    0.0|  

In [18]:
#Operation 2 — Select specific columns using select()
df.select(
    "date",
    "country",
    "daily_new_cases",
    "daily_new_deaths"
).show(10)

+-------------------+-----------+---------------+----------------+
|               date|    country|daily_new_cases|daily_new_deaths|
+-------------------+-----------+---------------+----------------+
|2020-02-15 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-16 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-17 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-18 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-19 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-20 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-21 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-22 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-23 00:00:00|Afghanistan|           NULL|            NULL|
|2020-02-24 00:00:00|Afghanistan|           NULL|            NULL|
+-------------------+-----------+---------------+----------------+
only showing top 10 rows


In [21]:
#Operation 3 — Filter records using filter()
df.filter(
    F.col("daily_new_cases") > 0
).select(
    "date",
    "country",
    "daily_new_cases"
).show(10)

+-------------------+-----------+---------------+
|               date|    country|daily_new_cases|
+-------------------+-----------+---------------+
|2020-03-07 00:00:00|Afghanistan|            3.0|
|2020-03-10 00:00:00|Afghanistan|            1.0|
|2020-03-11 00:00:00|Afghanistan|            2.0|
|2020-03-14 00:00:00|Afghanistan|            4.0|
|2020-03-15 00:00:00|Afghanistan|            5.0|
|2020-03-16 00:00:00|Afghanistan|            5.0|
|2020-03-17 00:00:00|Afghanistan|            1.0|
|2020-03-20 00:00:00|Afghanistan|            2.0|
|2020-03-22 00:00:00|Afghanistan|           16.0|
|2020-03-24 00:00:00|Afghanistan|           34.0|
+-------------------+-----------+---------------+
only showing top 10 rows


In [24]:
#Operation 4 — Group and aggregate using groupBy() and sum()
country_cases = (
    df.groupBy("country")
      .agg(
          F.sum("daily_new_cases").alias("total_reported_new_cases")
      )
      .orderBy(
          F.desc("total_reported_new_cases")
      )
)

country_cases.show(10, truncate=False)

+-----------+------------------------+
|country    |total_reported_new_cases|
+-----------+------------------------+
|USA        |8.4209458E7             |
|India      |4.3121596E7             |
|Brazil     |3.0682093E7             |
|France     |2.916079E7              |
|Germany    |2.5774493E7             |
|UK         |2.2159772E7             |
|Russia     |1.8255335E7             |
|South Korea|1.7756599E7             |
|Italy      |1.7030708E7             |
|Turkey     |1.5053165E7             |
+-----------+------------------------+
only showing top 10 rows


In [25]:
df.createOrReplaceTempView("covid_data")

print("Temporary SQL view created!")

Temporary SQL view created!


In [29]:
#Operation 1 — Retrieve records using SELECT and LIMIT
spark.sql("""
    SELECT
        date,
        country,
        daily_new_cases,
        daily_new_deaths
    FROM covid_data
    LIMIT 10
""").show(truncate=False)

+-------------------+-----------+---------------+----------------+
|date               |country    |daily_new_cases|daily_new_deaths|
+-------------------+-----------+---------------+----------------+
|2020-02-15 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-16 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-17 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-18 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-19 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-20 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-21 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-22 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-23 00:00:00|Afghanistan|NULL           |NULL            |
|2020-02-24 00:00:00|Afghanistan|NULL           |NULL            |
+-------------------+-----------+---------------+----------------+



In [37]:
#Operation 2 — Filter records using WHERE
spark.sql("""
    SELECT
        date,
        country,
        daily_new_cases
    FROM covid_data
    WHERE daily_new_cases > 0
    LIMIT 10
""").show(truncate=False)

+-------------------+-----------+---------------+
|date               |country    |daily_new_cases|
+-------------------+-----------+---------------+
|2020-03-07 00:00:00|Afghanistan|3.0            |
|2020-03-10 00:00:00|Afghanistan|1.0            |
|2020-03-11 00:00:00|Afghanistan|2.0            |
|2020-03-14 00:00:00|Afghanistan|4.0            |
|2020-03-15 00:00:00|Afghanistan|5.0            |
|2020-03-16 00:00:00|Afghanistan|5.0            |
|2020-03-17 00:00:00|Afghanistan|1.0            |
|2020-03-20 00:00:00|Afghanistan|2.0            |
|2020-03-22 00:00:00|Afghanistan|16.0           |
|2020-03-24 00:00:00|Afghanistan|34.0           |
+-------------------+-----------+---------------+



In [35]:
#Operation 3 — Group, aggregate, and sort using SQL
spark.sql("""
    SELECT
        country,
        SUM(daily_new_cases)::numeric AS total_reported_new_cases
    FROM covid_data
    GROUP BY country
    ORDER BY total_reported_new_cases DESC
    LIMIT 10
""").show(truncate=False)

+-----------+------------------------+
|country    |total_reported_new_cases|
+-----------+------------------------+
|USA        |84209458                |
|India      |43121596                |
|Brazil     |30682093                |
|France     |29160790                |
|Germany    |25774493                |
|UK         |22159772                |
|Russia     |18255335                |
|South Korea|17756599                |
|Italy      |17030708                |
|Turkey     |15053165                |
+-----------+------------------------+



In [34]:
spark.sql("""
    SELECT
        country,
        SUM(daily_new_cases)::numeric AS total_reported_new_cases
    FROM covid_data
    GROUP BY country
    ORDER BY total_reported_new_cases DESC
    LIMIT 1
""").show(truncate=False)

+-------+------------------------+
|country|total_reported_new_cases|
+-------+------------------------+
|USA    |84209458                |
+-------+------------------------+



In [38]:
df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show(truncate=False)

+----+-------+----------------------+---------------+------------+-----------------------+----------------+
|date|country|cumulative_total_cases|daily_new_cases|active_cases|cumulative_total_deaths|daily_new_deaths|
+----+-------+----------------------+---------------+------------+-----------------------+----------------+
|0   |0      |0                     |10458          |18040       |6560                   |26937           |
+----+-------+----------------------+---------------+------------+-----------------------+----------------+

